<a href="https://colab.research.google.com/github/RasheedHuma/RasheedHuma/blob/main/segment_transcripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transcript Segmentation Notebook

This notebook:
1. Loads a CSV where each row is one transcript.
2. Creates a unique transcript identifier.
3. Splits `main text` into segments and explodes into one row per segment.
4. Saves the result back to Google Drive.


In [ ]:
import pandas as pd
import re
import hashlib
from typing import List

## 1) Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2) Set the path to your CSV in Drive


In [ ]:
# Read in transcript data
INPUT_CSV_PATH = '/content/drive/MyDrive/AD Content Analysis/Data compilation and cleaning/AD_News_All_Unsegmented.csv'

CHANNEL_COL = 'Channel'
TEXT_COL = 'Body'

df = pd.read_csv(INPUT_CSV_PATH)
df.count() # Count values in each column

,0
Unnamed: 0,1279
Filename,1279
Channel,1248
Show,1248
Date,1279
Words,1279
Body,1279


In [ ]:
# See how many rows don't have channel name
na_count = df['Channel'].isna().sum()
print("Rows that will be removed:", na_count)

Rows that will be removed: 31


In [ ]:
# Remove rows without channel name
df.dropna(subset=['Channel'], inplace=True)
df.count()

,0
Unnamed: 0,1248
Filename,1248
Channel,1248
Show,1248
Date,1248
Words,1248
Body,1248
segments,1248


In [ ]:
# Create ID column
df.rename(columns={df.columns[0]: "Transcript_ID"}, inplace=True)
df.head(10)

,Transcript_ID,Filename,Channel,Show,Date,Words,Body,segments
0,0,Copy of $18 Billion Powerball Jackpot Won On C...,CNN,News Central,"December 26, 2025",3232,"RAFAEL ROMO, CNN CORRESPONDENT: Wednesday, eig...","[RAFAEL ROMO, CNN CORRESPONDENT: Wednesday, ei..."
1,1,Copy of ' New York Times ' Sues Pentagon ; New...,CNN,THE SITUATION ROOM,"December 4, 2025",3285,"WOLF BLITZER, CNN HOST: Bovino spoke about the...","[WOLF BLITZER, CNN HOST: Bovino spoke about th..."
2,2,"Copy of 11 NOLA Jail Inmates Escaped, At Least...",Fox News,THE STORY WITH MARTHA MACCALLUM,"May 18, 2025",8811,"(COMMERCIAL BREAK)\nMARTHA MACCALLUM, FOX NEWS...","[MARTHA MACCALLUM, FOX NEWS ANCHOR: All right...."
3,3,Copy of 15 Days Until The Election.PDF,Fox News,FOX HANNITY,"October 21, 2024",7949,"SEAN HANNITY, FOX NEWS HOST: And tonight, with...","[SEAN HANNITY, FOX NEWS HOST: And tonight, wit..."
4,4,Copy of 19 Suspected TDA Members Detained In A...,Fox News,@ NIGHT,"December 18, 2024",7409,"KAT TIMPF, FOX NEWS CONTRIBUTOR: Thanks to Joe...","[KAT TIMPF, FOX NEWS CONTRIBUTOR: Thanks to Jo..."
5,5,Copy of 20+ Dem Governors Join Biden In _Crisi...,Fox News,@ NIGHT,"July 3, 2024",7392,"(COMMERCIAL BREAK)\nGREG GUTFELD, FOX NEWS CHA...","[GREG GUTFELD, FOX NEWS CHANNEL HOST: Out of t..."
7,7,"Copy of 48 Hours - Deputy Spivey on Trial, CBS...",CBS News,News Transcripts Cbs 48 Hours,"January 11, 2025",7062,"(ANNOUNCEMENT)\n(Graphic onscreen: Houston, Te...","[(ANNOUNCEMENT)\n(Graphic onscreen: Houston, T..."
8,8,Copy of 48 Hours - Rodney Alcala_ The Killing ...,CBS News,News Transcripts Cbs 48 Hours,"June 1, 2024",6435,(ANNOUNCEMENT)\n(Graphic onscreen: One of the ...,[(ANNOUNCEMENT)\n(Graphic onscreen: One of the...
9,9,Copy of 49ers' Ricky Pearsall To Miss Multiple...,Fox News,THE GREG GUTFELD SHOW,"September 3, 2024",3071,(COMMERCIAL BREAK)\n(BEGIN VIDEO CLIP)\nUNIDEN...,[(BEGIN VIDEO CLIP)\nUNIDENTIFIED MALE: A stor...
10,10,Copy of 6 House Dems Say Biden Should Leave Ra...,CNN,ANDERSON COOPER 360 DEGREES,"July 8, 2024",8364,"MIKE BOYLAN, STORM CHASER: Actually June, righ...","[MIKE BOYLAN, STORM CHASER: Actually June, rig..."


In [ ]:
# Create and run function to split transcripts into segments
CHANNEL_MARKERS = {
    'CNN': r'\(commercial\s+break\)',
    'FOX NEWS': r'\(commercial\s+break\)',
    'ABC NEWS': r'\((ANNOUNCEMENTS|COMMERCIAL\s+BREAK)\)',
    'CBS NEWS': r'\(ANNOUNCEMENTS\)'
}

def split_into_segments(channel: str, text: str) -> List[str]:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return []
    text = str(text)
    if channel is None or (isinstance(channel, float) and pd.isna(channel)):
        return [text.strip()] if text.strip() else []

    ch = str(channel).strip().upper()
    pattern = CHANNEL_MARKERS.get(ch)
    if not pattern:
        return [text.strip()] if text.strip() else []

    parts = re.split(pattern, text, flags=re.IGNORECASE)
    parts = [p.strip() for p in parts if p and p.strip()]
    return parts

df['segments'] = df.apply(lambda r: split_into_segments(r[CHANNEL_COL], r[TEXT_COL]), axis=1)

seg_df = df.explode('segments', ignore_index=True)
seg_df = seg_df[seg_df['segments'].notna() & (seg_df['segments'].astype(str).str.strip() != '')].copy()

seg_df['segment_number'] = seg_df.groupby('Transcript_ID').cumcount() + 1

seg_df[TEXT_COL] = seg_df['segments']
seg_df = seg_df.drop(columns=['segments'])

# Count number of words in each segment
seg_df['segment_words'] = seg_df[TEXT_COL].astype(str).str.split().str.len()

# Drop segments with fewer than 10 words
seg_df = seg_df[seg_df['segment_words'] >= 50].copy()

print('Original transcripts:', len(df))
print('Segmented rows:', len(seg_df))
seg_df.head(50)


Original transcripts: 1248
Segmented rows: 5180


,Transcript_ID,Filename,Channel,Show,Date,Words,Body,segment_number,segment_words
0,0,Copy of $18 Billion Powerball Jackpot Won On C...,CNN,News Central,"December 26, 2025",3232,"RAFAEL ROMO, CNN CORRESPONDENT: Wednesday, eig...",1,183
1,0,Copy of $18 Billion Powerball Jackpot Won On C...,CNN,News Central,"December 26, 2025",3232,BOLDUAN: This just in. We are now -- we've now...,2,1149
2,0,Copy of $18 Billion Powerball Jackpot Won On C...,CNN,News Central,"December 26, 2025",3232,"SIDNER: All right. On our radar for you today,...",3,1890
4,1,Copy of ' New York Times ' Sues Pentagon ; New...,CNN,THE SITUATION ROOM,"December 4, 2025",3285,"WOLF BLITZER, CNN HOST: Bovino spoke about the...",1,1368
5,1,Copy of ' New York Times ' Sues Pentagon ; New...,CNN,THE SITUATION ROOM,"December 4, 2025",3285,"BLITZER: All right, back to our breaking news,...",2,1267
6,1,Copy of ' New York Times ' Sues Pentagon ; New...,CNN,THE SITUATION ROOM,"December 4, 2025",3285,"BLITZER: New this morning, ""The New York Times...",3,646
7,2,"Copy of 11 NOLA Jail Inmates Escaped, At Least...",Fox News,THE STORY WITH MARTHA MACCALLUM,"May 18, 2025",8811,"MARTHA MACCALLUM, FOX NEWS ANCHOR: All right. ...",1,1390
8,2,"Copy of 11 NOLA Jail Inmates Escaped, At Least...",Fox News,THE STORY WITH MARTHA MACCALLUM,"May 18, 2025",8811,"MACCALLUM: I mean, this one's getting a lot of...",2,1362
9,2,"Copy of 11 NOLA Jail Inmates Escaped, At Least...",Fox News,THE STORY WITH MARTHA MACCALLUM,"May 18, 2025",8811,"MACCALLUM: So, his new claims around the, quot...",3,1940
10,2,"Copy of 11 NOLA Jail Inmates Escaped, At Least...",Fox News,THE STORY WITH MARTHA MACCALLUM,"May 18, 2025",8811,MACCALLUM: Bit of a setback on this Friday aft...,4,4111


In [ ]:
seg_df.describe()

,Transcript_ID,Words,segment_number,segment_words
count,5180.000000,5180.000000,5180.000000,5180.000000
mean,632.005019,7509.992471,3.576834,1411.448649
std,368.640281,3070.756079,3.018781,1119.735721
min,0.000000,103.000000,1.000000,50.000000
25%,297.750000,6775.000000,2.000000,724.000000
50%,636.000000,7531.000000,3.000000,1159.000000
75%,942.000000,8253.000000,4.000000,1763.000000
max,1278.000000,19183.000000,19.000000,16625.000000


In [ ]:
#Some of the segments are still very long (15k words). Let's break them down into smaller than 1000 words segments.
import re
import math

def split_long_segments(text, max_words=1000):
    if not text or not str(text).strip():
        return []

    text = re.sub(r'\s+', ' ', str(text).strip())
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

    if not sentences:
        return [text]

    total_words = sum(len(s.split()) for s in sentences)

    # do nothing if already small
    if total_words <= max_words:
        return [text]

    # number of chunks
    n_chunks = math.ceil(total_words / max_words)
    target = total_words / n_chunks

    chunks = []
    current = []
    current_words = 0

    for sent in sentences:
        sent_words = len(sent.split())

        # don't exceed max_words
        if current and current_words + sent_words > max_words:
            chunks.append(" ".join(current))
            current = [sent]
            current_words = sent_words
            continue

        # balance chunk sizes
        if current and current_words >= target:
            chunks.append(" ".join(current))
            current = [sent]
            current_words = sent_words
            continue

        current.append(sent)
        current_words += sent_words

    if current:
        chunks.append(" ".join(current))

    return chunks

In [ ]:
seg_df['segment_words'] = seg_df[TEXT_COL].str.split().str.len()

seg_df['split_parts'] = seg_df.apply(
    lambda r: split_long_segments(r[TEXT_COL], max_words=1000)
    if r['segment_words'] > 1000
    else [r[TEXT_COL]],
    axis=1
)

seg_df = seg_df.explode('split_parts', ignore_index=True)

seg_df[TEXT_COL] = seg_df['split_parts']
seg_df = seg_df.drop(columns=['split_parts'])

seg_df['segment_words'] = seg_df[TEXT_COL].str.split().str.len()

# drop small segments
seg_df = seg_df[seg_df['segment_words'] >= 100].copy()

# reset index (optional but clean)
seg_df = seg_df.reset_index(drop=True)

In [ ]:
seg_df['segment_words'].describe()

,segment_words
count,9899.000000
mean,737.407011
std,177.760957
min,100.000000
25%,623.000000
50%,766.000000
75%,873.000000
max,1000.000000


In [ ]:
#Create a segment unique ID
seg_df["Segment_ID"] = range(1, len(seg_df) + 1)

In [ ]:
#Identify AD related segments

import re

KEYWORDS = {
    "alzheimer",
    "dementia",
    "cognitive decline",
    "cognitive impairment",
    "memory loss",
}

pattern = "|".join(re.escape(word) for word in KEYWORDS)

seg_df["AD"] = (
    seg_df["Body"]
    .fillna("")
    .str.lower()
    .str.contains(pattern, regex=True)
    .astype(int)
)

In [ ]:
seg_df.columns

Index(['Transcript_ID', 'Filename', 'Channel', 'Show', 'Date', 'Words', 'Body',
       'segment_number', 'segment_words', 'Segment_ID', 'AD'],
      dtype='object')

In [ ]:
seg_df = seg_df[["Segment_ID", "Transcript_ID", "Filename", "Channel", "Show", "Date", "Body", "segment_words","AD"]].copy()

In [ ]:
seg_df.head(30)

,Segment_ID,Transcript_ID,Filename,Channel,Show,Date,Body,segment_words,AD
0,1,0,Copy of $18 Billion Powerball Jackpot Won On C...,CNN,News Central,"December 26, 2025","RAFAEL ROMO, CNN CORRESPONDENT: Wednesday, eig...",183,0
1,2,0,Copy of $18 Billion Powerball Jackpot Won On C...,CNN,News Central,"December 26, 2025",BOLDUAN: This just in. We are now -- we've now...,575,0
2,3,0,Copy of $18 Billion Powerball Jackpot Won On C...,CNN,News Central,"December 26, 2025","So in reality, how far does a round of strikes...",574,0
3,4,0,Copy of $18 Billion Powerball Jackpot Won On C...,CNN,News Central,"December 26, 2025","SIDNER: All right. On our radar for you today,...",956,0
4,5,0,Copy of $18 Billion Powerball Jackpot Won On C...,CNN,News Central,"December 26, 2025","TIM ANDREWS: It may shorten your life, but you...",934,1
5,6,1,Copy of ' New York Times ' Sues Pentagon ; New...,CNN,THE SITUATION ROOM,"December 4, 2025","WOLF BLITZER, CNN HOST: Bovino spoke about the...",688,0
6,7,1,Copy of ' New York Times ' Sues Pentagon ; New...,CNN,THE SITUATION ROOM,"December 4, 2025",I have heard from our business owners in the c...,680,1
7,8,1,Copy of ' New York Times ' Sues Pentagon ; New...,CNN,THE SITUATION ROOM,"December 4, 2025","BLITZER: All right, back to our breaking news,...",674,0
8,9,1,Copy of ' New York Times ' Sues Pentagon ; New...,CNN,THE SITUATION ROOM,"December 4, 2025","Many people do. It seems crazy, but the FBI ha...",593,0
9,10,1,Copy of ' New York Times ' Sues Pentagon ; New...,CNN,THE SITUATION ROOM,"December 4, 2025","BLITZER: New this morning, ""The New York Times...",646,0


## 3) Save output back to Google Drive


In [ ]:
OUTPUT_CSV_PATH = '/content/drive/MyDrive/AD Content Analysis/Data compilation and cleaning/segmented_transcripts_06252026.csv'
seg_df.to_csv(OUTPUT_CSV_PATH, index=False)
print('Saved to:', OUTPUT_CSV_PATH)

Saved to: /content/drive/MyDrive/AD Content Analysis/Data compilation and cleaning/segmented_transcripts_06252026.csv
